# Evaluación Sumativa Unidad 2-Isaac Vire

# Apartado 1

In [ ]:
import pandas as pd
from scipy import stats

# Carga del dataset regional
nombres_columnas = ["Fecha", "Tipo", "Emisor", "Cantidad", "Valor_Nominal",
                    "Precio", "Valor_Nominal_Total", "Monto_Efectivo",
                    "Casa_Origen", "Casa_Destino", "Bolsa"]

df = pd.read_csv("BVG_Acciones_export.csv", names=nombres_columnas, skiprows=1)
variable_critica = df['Precio'].dropna()

# Definición de Hipótesis
# H0: El precio promedio de la región es igual a $5 USD
# H1: El precio promedio de la región es DIFERENTE a $5 USD
mu_0 = 5.00
alpha = 0.05

# Ejecución del test T de Student (Varianza poblacional desconocida)
t_stat, p_value = stats.ttest_1samp(variable_critica, popmean=mu_0)

# Valor-p
print(f"Estadístico T: {t_stat:.4f}")
print(f"Valor-p: {p_value:.4e}")

if p_value < alpha:
    print("Decisión: Se RECHAZA la hipótesis nula (H0).")
    print("Justificación: Existe evidencia suficiente para afirmar que el precio promedio difiere del valor histórico.")
else:
    print("Decisión: NO se rechaza la hipótesis nula (H0).")
    print("Justificación: No hay evidencia suficiente para afirmar un cambio significativo; la diferencia es probable por azar.")

Estadístico T: 26.6776
Valor-p: 6.2508e-155
Decisión: Se RECHAZA la hipótesis nula (H0).
Justificación: Existe evidencia suficiente para afirmar que el precio promedio difiere del valor histórico.


## Como se sabe dado los conocimientos previos adquiridos por los APEs realizados, cuando p < 0.05, se rechaza que H0 es falsa por el hecho de que hay datos, resultados, que no son dados por el azar. Se preficere usar T de Student por la varianza desconocida, pero ante tantos datos como los hay en el dataset utilizado, se puede usar Z por el teorema del limite central, aunque dará el mismo resultado. Los grados de libertad calculados ayudan a cubrir la incertidumbre dada por no conocer la varianza y asegurar más precisión.

# Apartado 2

In [ ]:
import pandas as pd
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from IPython.display import display

# Carga y limpieza
nombres_columnas = ["Fecha", "Tipo", "Emisor", "Cantidad", "Valor_Nominal",
                    "Precio", "Valor_Nominal_Total", "Monto_Efectivo",
                    "Casa_Origen", "Casa_Destino", "Bolsa"]
df = pd.read_csv("BVG_Acciones_export.csv", names=nombres_columnas, skiprows=1)
df_limpio = df[['Precio', 'Emisor']].dropna()

# Los 10 emisores más frecuentes
top_10_emisores = df_limpio['Emisor'].value_counts().nlargest(10).index
df_10Emisores = df_limpio[df_limpio['Emisor'].isin(top_10_emisores)]

# Ejecución de ANOVA de 1 factor
grupos = [group['Precio'].values for name, group in df_10Emisores.groupby('Emisor')]
f_stat, p_value_anova = stats.f_oneway(*grupos)

print(f"Estadístico F de ANOVA: {f_stat:.4f}")
print(f"Valor-p de ANOVA: {p_value_anova:.4e}")

# Prueba Post-Hoc de Tukey
if p_value_anova < 0.05:
    print("\n--- Comparaciones Específicas (Top 10 Emisores Principales) ---")
    tukey = pairwise_tukeyhsd(endog=df_10Emisores['Precio'],
                              groups=df_10Emisores['Emisor'],
                              alpha=0.05)
    # Se imprime el resumen con todas las comparaciones por pares
    display(tukey.summary())
else:
    print("\nNo existen diferencias significativas entre las medias de los emisores analizados.")

Estadístico F de ANOVA: 2292.5206
Valor-p de ANOVA: 0.0000e+00

--- Comparaciones Específicas (Top 10 Emisores Principales) ---


group1,group2,meandiff,p-adj,lower,upper,reject
BANCO DE LA PRODUCCION S.A PRODUBANCO,BANCO GUAYAQUIL S.A.,0.2671,1.0,-16.9075,17.4417,False
BANCO DE LA PRODUCCION S.A PRODUBANCO,BANCO PICHINCHA C.A.,89.316,0.0,66.0171,112.6149,True
BANCO DE LA PRODUCCION S.A PRODUBANCO,BIENES RAÍCES E INVERSIONES DE CAPITAL BRIKAPITAL S.A.,900.5786,0.0,873.8718,927.2855,True
BANCO DE LA PRODUCCION S.A PRODUBANCO,BOLSA DE VALORES DE QUITO BVQ SOCIEDAD ANÓNIMA,1.359,1.0,-25.3617,28.0798,False
BANCO DE LA PRODUCCION S.A PRODUBANCO,CERVECERIA NACIONAL CN S.A.,58.2753,0.0,34.2816,82.2689,True
BANCO DE LA PRODUCCION S.A PRODUBANCO,CONJUNTO CLINICO NACIONAL CONCLINA S.A.,629.7228,0.0,600.7133,658.7322,True
BANCO DE LA PRODUCCION S.A PRODUBANCO,CORPORACION FAVORITA C.A.,1.1713,1.0,-13.6269,15.9694,False
BANCO DE LA PRODUCCION S.A PRODUBANCO,HOLCIM ECUADOR S.A.,48.7479,0.0,21.5805,75.9153,True
BANCO DE LA PRODUCCION S.A PRODUBANCO,NATLUK S.A.,6.257,0.9993,-20.8361,33.3501,False
BANCO GUAYAQUIL S.A.,BANCO PICHINCHA C.A.,89.0489,0.0,67.8662,110.2316,True


## El ANOVA compara la media tres o más grupos que son independientes, utiliza el estadístico F que se crea de la relación entre la varianza dentro de los grupos y la varianza entre los grupos. Siendo un F grande y un valor p pequeño muestra de que hay diferencias entre las medias de algunos grupos. Se contempla el Error Estándar para tener más precisión y grados de libertad entre grupos (k-1) y grados de libertad dentro de grupos (N-k) para más precisión con el estadístico F y el valor p. La prueba Post-Hoc de Tukey se utiliza para saber, en caso de que el ANOVA indique diferencias, entre qué pares de medias existen esas diferencias y cuál es su magnitud, realizando comparaciones ajustadas para múltiples pruebas.

## El A/B Testing se usa para comparar dos versiones de un proceso o producto para saber cuál es mejor, sus resultados están manipulados a propósito y aleatorizados para que se tenga una métrica que solo dependa de la manipulación y nada más, por lo que el dataset no es apropiado para eso.